<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day03-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 3 — In-class discussion problem (1 of 3)

Work this out **by hand in your group first** — then run the code cell to check your answer before presenting.

## What level is this accession at?

Four real accessions: one whole chromosome, two records from the same gene, and one predicted record from an unrelated gene:

1. `NC_000017.11`
2. `NM_000546.6`
3. `NP_000537.3`
4. `XP_011536997.1`

For each one:

1. Name its **level** — genomic (DNA), mRNA (RNA), or protein — from the prefix alone.
2. Say whether it is a **known/curated** record or a **predicted** one.
3. `NM_000546.6` and `NP_000537.3` turn out to be the mRNA and protein for the *same* gene. Which gene, and how would you confirm the two records actually belong together rather than just happening to share a similar number?

In [1]:
from Bio import Entrez, SeqIO
Entrez.email = "kb8029-book@example.org"

REFSEQ_PREFIXES = {
    "NC": "genomic (known)", "NM": "mRNA (known)", "NP": "protein (known)",
    "NR": "non-coding RNA (known)", "XM": "mRNA (predicted)", "XP": "protein (predicted)",
}

accessions = ["NC_000017.11", "NM_000546.6", "NP_000537.3", "XP_011536997.1"]
dbs = {"NC": "nucleotide", "NM": "nucleotide", "NP": "protein", "XM": "nucleotide", "XP": "protein"}

for acc in accessions:
    prefix = acc.split("_")[0]
    handle = Entrez.efetch(db=dbs[prefix], id=acc, rettype="gb", retmode="text")
    rec = SeqIO.read(handle, "genbank")
    handle.close()
    print(f"{acc:16s} {REFSEQ_PREFIXES[prefix]:22s} -> {rec.description}")

NC_000017.11     genomic (known)        -> Homo sapiens chromosome 17, GRCh38.p14 Primary Assembly


NM_000546.6      mRNA (known)           -> Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA


NP_000537.3      protein (known)        -> cellular tumor antigen p53 isoform a [Homo sapiens]


XP_011536997.1   protein (predicted)    -> SWI/SNF-related matrix-associated actin-dependent regulator of chromatin subfamily D member 1 isoform X3 [Homo sapiens]


**Discussion point:** `NC_000017.11` is genomic (human chromosome 17), `NM_000546.6` is the mRNA for *TP53* (tumor protein p53), `NP_000537.3` is *TP53*'s protein, and `XP_011536997.1` is a *predicted* protein from a computational annotation pipeline — its X prefix means nobody has directly confirmed this exact protein sequence experimentally, unlike the N-prefixed records above it.

The confirmation that `NM_000546.6` and `NP_000537.3` belong together isn't guesswork: the protein record's own `DBSOURCE` field states it explicitly.

In [2]:
from Bio import Entrez, SeqIO
Entrez.email = "kb8029-book@example.org"

handle = Entrez.efetch(db="protein", id="NP_000537.3", rettype="gb", retmode="text")
protein = SeqIO.read(handle, "genbank")
handle.close()

print("Cross-reference:", protein.annotations["db_source"])
assert "NM_000546" in protein.annotations["db_source"]

Cross-reference: REFSEQ: accession NM_000546.6
